In [1]:
import qiskit
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
from qiskit.visualization import plot_histogram,plot_distribution
from qiskit.quantum_info import Statevector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService
from qiskit_aer import Aer, AerSimulator
from qiskit.circuit.library import UGate
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt
import pylatexenc
import numpy as np
import math
import os

In [16]:
api_key = os.getenv("API_KEY")
crn = os.getenv("CRN")
QiskitRuntimeService.save_account(
token = api_key, # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
instance = crn, # Optional
overwrite = True 
)
service = QiskitRuntimeService()
service.saved_accounts() # To chek the account is saved successfully
# backend = service.least_busy(
#     operational=True, simulator=False, min_num_qubits=127
# )
# backend.name
backend_name = "ibm_kingston" 
backend = service.backend(backend_name)

In [17]:
def build_circuit(eps, troter_steps):
    # DEFINE VARIABLES
    gup_coeff = 0.001
    w0 = 1 - (2 * gup_coeff)
    w1 = math.sqrt(2) - (4 * gup_coeff)
    u0 = 1
    u1 = math.sqrt(2)
    theta_1 = (w0 * u0 * eps)
    theta_2 = (w0 * u1 * eps)
    theta_3 = (w1 * u0 * eps)
    theta_4 = (w1 * u1 * eps)
    # print(theta_1, theta_2, theta_3, theta_4)
    # FOR TROTTERIZATION
    dtheta_1 = theta_1 / troter_steps
    dtheta_2 = theta_2 / troter_steps
    dtheta_3 = theta_3 / troter_steps
    dtheta_4 = theta_4 / troter_steps
    b0 = QuantumRegister(1, "b0")
    b1 = QuantumRegister(1,"b1")
    b2 = QuantumRegister(1,"b2")

    a0 = QuantumRegister(1, "a0")
    a1 = QuantumRegister(1,"a1")
    a2 = QuantumRegister(1,"a2")
    creg = ClassicalRegister(6,name = "measurement")
    quantum_circuit = QuantumCircuit(b0,a0,b1,a1,b2,a2,creg)
    # INITIAL STATE PREPARATION
    alpha = 1 / np.sqrt(1 + (gup_coeff**2)/2)
    beta  = (gup_coeff / np.sqrt(2)) / np.sqrt(1 + (gup_coeff**2)/2)
    # exact rotation angle
    theta_prep = 2 * np.arcsin(beta)
    quantum_circuit.x(0)  # b-register vacuum |100>
    quantum_circuit.x(1)  # a-register vacuum |100>
    quantum_circuit.cry(theta_prep, 0, 4)
    quantum_circuit.cx(4, 0)
    # FIRST BLOCK H00
    # STRING 1
    for _ in range(troter_steps):
        quantum_circuit.cx(b0,a1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a0)
        quantum_circuit.s(b0)
        quantum_circuit.rx(np.pi/2, b0)

        quantum_circuit.p(dtheta_1,b0)

        quantum_circuit.rx(-np.pi/2, b0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a1)
        # barrier
        # quantum_circuit.barrier(label = "1.2")
        # STRING 2
        quantum_circuit.s(a1)
        quantum_circuit.s(a0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a0)
        quantum_circuit.s(b0)
        quantum_circuit.rx(np.pi/2, b0)

        quantum_circuit.p(dtheta_1,b0)

        quantum_circuit.rx(-np.pi/2, b0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.sdg(a0)
        quantum_circuit.sdg(a1)
        # barrier
        # quantum_circuit.barrier(label = "2.3")
        # STRING 3
        quantum_circuit.s(b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a0)
        quantum_circuit.s(b0)
        quantum_circuit.rx(np.pi/2,b0)

        quantum_circuit.p(dtheta_1, b0)

        quantum_circuit.rx(-np.pi/2,b0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.sdg(b0)
        quantum_circuit.sdg(b1)
        # barrier
        # quantum_circuit.barrier(label = "3.4")
        # STRING 4
        quantum_circuit.s(a1)
        quantum_circuit.s(b1)
        quantum_circuit.s(a0)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a0)
        quantum_circuit.s(b0)
        quantum_circuit.rx(np.pi/2,b0)

        quantum_circuit.p(dtheta_1, b0)

        quantum_circuit.rx(-np.pi/2,b0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.sdg(b0)
        quantum_circuit.sdg(a0)
        quantum_circuit.sdg(b1)
        quantum_circuit.sdg(a1)
        # Barrier
        # quantum_circuit.barrier(label = "BLOCK 2")
        # BLOCK 2
        # STRING 1
        quantum_circuit.cx(b0,a2)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.s(b0)
        quantum_circuit.rx(np.pi/2, b0)

        quantum_circuit.p(dtheta_2,b0)

        quantum_circuit.rx(-np.pi/2, b0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a2)
        # BARRIER
        # quantum_circuit.barrier(label = "1.2")
        # STRING 2
        quantum_circuit.s(a2)
        quantum_circuit.s(a1)
        quantum_circuit.cx(b0,a2)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.s(b0)
        quantum_circuit.rx(np.pi/2, b0)

        quantum_circuit.p(dtheta_2,b0)

        quantum_circuit.rx(-np.pi/2, b0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a2)
        quantum_circuit.sdg(a1)
        quantum_circuit.sdg(a2)
        # BARRIER 
        # quantum_circuit.barrier(label = "2.3")
        # STRING 3
        quantum_circuit.s(b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a2)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.s(b0)
        quantum_circuit.rx(np.pi/2,b0)

        quantum_circuit.p(dtheta_2, b0)

        quantum_circuit.rx(-np.pi/2,b0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a2)
        quantum_circuit.sdg(b0)
        quantum_circuit.sdg(b1)
        # BARRIER 
        # quantum_circuit.barrier(label = "3.4")
        # STRING 4
        quantum_circuit.s(a2)
        quantum_circuit.s(b1)
        quantum_circuit.s(a1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a2)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.s(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.s(b0)
        quantum_circuit.rx(np.pi/2,b0)

        quantum_circuit.p(dtheta_2, b0)

        quantum_circuit.rx(-np.pi/2,b0)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,b1)
        quantum_circuit.sdg(b0)
        quantum_circuit.cx(b0,a2)
        quantum_circuit.sdg(b0)
        quantum_circuit.sdg(a1)
        quantum_circuit.sdg(b1)
        quantum_circuit.sdg(a2)
        # BARRIER
        # quantum_circuit.barrier(label = "Block 3")
        # BLOCK 3
        # STRING 1
        quantum_circuit.cx(b1,a1)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a0)
        quantum_circuit.s(b1)
        quantum_circuit.rx(np.pi/2, b1)

        quantum_circuit.p(dtheta_3,b1)

        quantum_circuit.rx(-np.pi/2, b1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a0)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a1)
        # barrier
        # quantum_circuit.barrier(label = "1.2")
        # STRING 2
        quantum_circuit.s(a1)
        quantum_circuit.s(a0)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a0)
        quantum_circuit.s(b1)
        quantum_circuit.rx(np.pi/2, b1)

        quantum_circuit.p(dtheta_3,b1)

        quantum_circuit.rx(-np.pi/2, b1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a0)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.sdg(a0)
        quantum_circuit.sdg(a1)
        # barrier
        # quantum_circuit.barrier(label = "2.3")
        # STRING 3
        quantum_circuit.s(b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a0)
        quantum_circuit.s(b1)
        quantum_circuit.rx(np.pi/2,b1)

        quantum_circuit.p(dtheta_3, b1)

        quantum_circuit.rx(-np.pi/2,b1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a0)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.sdg(b1)
        quantum_circuit.sdg(b2)
        # barrier
        # quantum_circuit.barrier(label = "3.4")
        # STRING 4
        quantum_circuit.s(a1)
        quantum_circuit.s(b2)
        quantum_circuit.s(a0)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a0)
        quantum_circuit.s(b1)
        quantum_circuit.rx(np.pi/2,b1)

        quantum_circuit.p(dtheta_3, b1)

        quantum_circuit.rx(-np.pi/2,b1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a0)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.sdg(b1)
        quantum_circuit.sdg(a0)
        quantum_circuit.sdg(b2)
        quantum_circuit.sdg(a1)
        # Barrier 
        # quantum_circuit.barrier(label = "Block 4")
        # BLOCK 4
        # STRING 1
        quantum_circuit.cx(b1,a2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.s(b1)
        quantum_circuit.rx(np.pi/2, b1)

        quantum_circuit.p(dtheta_4,b1)

        quantum_circuit.rx(-np.pi/2, b1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a2)
        # barrier
        # quantum_circuit.barrier(label = "1.2")
        # STRING 2
        quantum_circuit.s(a2)
        quantum_circuit.s(a1)
        quantum_circuit.cx(b1,a2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.s(b1)
        quantum_circuit.rx(np.pi/2, b1)

        quantum_circuit.p(dtheta_4,b1)

        quantum_circuit.rx(-np.pi/2, b1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a2)
        quantum_circuit.sdg(a1)
        quantum_circuit.sdg(a2)
        # barrier
        # quantum_circuit.barrier(label = "2.3")
        # STRING 3
        quantum_circuit.s(b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.s(b1)
        quantum_circuit.rx(np.pi/2,b1)

        quantum_circuit.p(dtheta_4, b1)

        quantum_circuit.rx(-np.pi/2,b1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a2)
        quantum_circuit.sdg(b1)
        quantum_circuit.sdg(b2)
        # barrier
        # quantum_circuit.barrier(label = "3.4")
        # STRING 4
        quantum_circuit.s(a2)
        quantum_circuit.s(b2)
        quantum_circuit.s(a1)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.s(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.s(b1)
        quantum_circuit.rx(np.pi/2,b1)

        quantum_circuit.p(dtheta_4, b1)

        quantum_circuit.rx(-np.pi/2,b1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a1)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,b2)
        quantum_circuit.sdg(b1)
        quantum_circuit.cx(b1,a2)
        quantum_circuit.sdg(b1)
        quantum_circuit.sdg(a1)
        quantum_circuit.sdg(b2)
        quantum_circuit.sdg(a2)
    quantum_circuit.measure([0,1,2,3,4,5],[0,1,2,3,4,5])
        # quantum_circuit.draw("mpl", fold = 51)
    return quantum_circuit
all_circuits = []
metadata = []
# eps_list = [0.001,0.002,0.003,0.004,0.005,0.006] # math.pow(10,-3), math.pow(10,-5) 
eps_list =  np.linspace(0.15, 0.25, 25)
troter_steps = [5]
for steps in troter_steps:
    for eps in eps_list:

        quantum_circuit = build_circuit(eps, steps)

        all_circuits.append(quantum_circuit)
        metadata.append((steps, eps))

# -----------------------------------
# 2️⃣ Transpile ALL circuits together
# -----------------------------------
transpiled_circuits = transpile(all_circuits, backend, optimization_level=3)

# -----------------------------------
# 3️⃣ Submit ONE job
# -----------------------------------
sampler = Sampler(mode=backend)

job = sampler.run(transpiled_circuits, shots=1024)
print("Job ID:", job.job_id())
print("Backend:", backend.name)
print("Status:", job.status())
job_result = job.result()

# -----------------------------------
# 4️⃣ Organize results nicely
# -----------------------------------
results = {}

for i, (steps, eps) in enumerate(metadata):

    counts = job_result[i].data.measurement.get_counts()

    if steps not in results:
        results[steps] = {}

    results[steps][eps] = counts

# -----------------------------------
# 5️⃣ Print clean output
# -----------------------------------
for steps in results:
    print("\n==============================")
    print("Trotter steps =", steps)
    print("==============================")

    for eps in results[steps]:
        print(f"eps = {eps} → {results[steps][eps]}")

Job ID: d7k6jda8ui0s73b4s88g
Backend: ibm_kingston
Status: QUEUED

Trotter steps = 5
eps = 0.15 → {'111010': 9, '001100': 18, '000011': 96, '000000': 11, '001010': 21, '001110': 18, '001011': 143, '101001': 10, '110110': 6, '100110': 22, '000010': 26, '001001': 29, '111101': 11, '101010': 19, '100010': 24, '000111': 31, '101011': 32, '011011': 24, '100101': 6, '101100': 13, '000001': 25, '011110': 8, '100111': 18, '110011': 14, '110101': 7, '110010': 12, '101111': 17, '111001': 11, '101110': 18, '011001': 14, '011101': 7, '111111': 9, '010001': 8, '001111': 29, '011111': 15, '001101': 13, '100000': 12, '010111': 9, '000100': 4, '000101': 11, '100011': 20, '010011': 17, '000110': 13, '110100': 6, '010000': 5, '010101': 10, '011010': 10, '111000': 6, '100001': 11, '111011': 10, '100100': 6, '110111': 10, '111110': 7, '001000': 6, '101000': 6, '010010': 10, '011100': 6, '110001': 4, '101101': 7, '111100': 6, '110000': 5, '010100': 4, '011000': 5, '010110': 4}
eps = 0.15416666666666667 → {

In [24]:
# def counts_to_probs(counts_dict):
#     shots = sum(counts_dict.values())
#     return {k: v / shots for k, v in counts_dict.items()}

# allowed = ['100100','100001','010010','001100','001001'] 

# # Convert counts → probabilities
# probs = counts_to_probs(results[steps][eps])

# # Compute total probability inside allowed subspace
# total = sum(probs.get(k, 0.0) for k in allowed)

# # Avoid division by zero
# if total == 0:
#     print("No allowed states measured!")
#     postselected_probs = {k: 0 for k in allowed}
# else:
#     postselected_probs = {k: probs.get(k, 0.0) / total for k in allowed}

# print(postselected_probs)

def counts_to_probs(counts_dict):
    shots = sum(counts_dict.values())
    return {k: v / shots for k, v in counts_dict.items()}

allowed = ['000011','110000','001100','100001','010010']

postselected_results = {}

for steps in results:
    postselected_results[steps] = {}

    for eps in results[steps]:
        counts = results[steps][eps]
        probs = counts_to_probs(counts)

        total = sum(probs.get(k, 0.0) for k in allowed)

        if total == 0:
            post_probs = {k: 0 for k in allowed}
        else:
            post_probs = {k: probs.get(k, 0.0) / total for k in allowed}

        postselected_results[steps][eps] = post_probs

        print(f"steps={steps}, eps={eps} → {post_probs}")

steps=5, eps=0.15 → {'000011': 0.6857142857142857, '110000': 0.03571428571428571, '001100': 0.12857142857142856, '100001': 0.07857142857142857, '010010': 0.07142857142857142}
steps=5, eps=0.15416666666666667 → {'000011': 0.4805194805194805, '110000': 0.05194805194805195, '001100': 0.14285714285714285, '100001': 0.2077922077922078, '010010': 0.11688311688311688}
steps=5, eps=0.15833333333333333 → {'000011': 0.834061135371179, '110000': 0.03056768558951965, '001100': 0.039301310043668124, '100001': 0.043668122270742356, '010010': 0.05240174672489083}
steps=5, eps=0.1625 → {'000011': 0.6741573033707865, '110000': 0.03932584269662921, '001100': 0.1348314606741573, '100001': 0.10112359550561797, '010010': 0.05056179775280899}
steps=5, eps=0.16666666666666666 → {'000011': 0.6195652173913043, '110000': 0.05434782608695652, '001100': 0.10869565217391304, '100001': 0.06521739130434782, '010010': 0.15217391304347827}
steps=5, eps=0.17083333333333334 → {'000011': 0.5185185185185185, '110000': 0.0

In [25]:
def compute_true_probabilities(eps_list, trotter_steps):
    true_results = {}

    for steps in trotter_steps:
        true_results[steps] = {}

        for eps in eps_list:

            # Build circuit
            qc = build_circuit(eps, steps)

            # Remove measurements (important!)
            qc_no_meas = qc.remove_final_measurements(inplace=False)

            # Get exact statevector
            state = Statevector.from_instruction(qc_no_meas)

            # Convert to probabilities
            probs = state.probabilities_dict()

            true_results[steps][eps] = probs

    return true_results
true_probs = compute_true_probabilities(eps_list, troter_steps)
print(true_probs)

{5: {np.float64(0.15): {np.str_('000011'): np.float64(0.9177023959752163), np.str_('001100'): np.float64(0.06276818417382166), np.str_('010010'): np.float64(0.00486741234458633), np.str_('011101'): np.float64(4.0427225555287843e-32), np.str_('100001'): np.float64(0.005109339409490861), np.str_('101110'): np.float64(7.658580794477366e-33), np.str_('110000'): np.float64(0.009552668096911538), np.str_('111111'): np.float64(2.9466728149124703e-32)}, np.float64(0.15416666666666667): {np.str_('000011'): np.float64(0.913483552131676), np.str_('001100'): np.float64(0.06492280602635721), np.str_('010010'): np.float64(0.005389749743119551), np.str_('011101'): np.float64(3.734498532788374e-33), np.str_('100001'): np.float64(0.005660868868399968), np.str_('101110'): np.float64(1.1675950287847616e-33), np.str_('110000'): np.float64(0.010543023230468879), np.str_('111111'): np.float64(8.647425450298689e-32)}, np.float64(0.15833333333333333): {np.str_('000011'): np.float64(0.9091926780145424), np.str

In [26]:
def classical_fidelity(p_true, p_exp):
    all_keys = set(p_true.keys()).union(set(p_exp.keys()))
    fidelity_sum = 0.0

    for key in all_keys:
        pt = p_true.get(key, 0.0)
        pe = p_exp.get(key, 0.0)
        fidelity_sum += np.sqrt(pt * pe)

    return fidelity_sum**2

fidelity_results = {}

for steps in results:
    fidelity_results[steps] = {}

    for eps in results[steps]:

        # Exact probabilities from statevector
        p_true = true_probs[steps][eps]

        # Hardware probabilities → apply postselection
        counts = results[steps][eps]
        p_raw = counts_to_probs(counts)

        # Apply postselection: keep only keys present in ideal probs
        allowed_keys = p_true.keys()
        total = sum(p_raw[k] for k in allowed_keys if k in p_raw)
        p_exp = {k: p_raw.get(k, 0.0)/total for k in allowed_keys}

        # Compute classical fidelity using postselected probs
        F = classical_fidelity(p_true, p_exp)

        fidelity_results[steps][eps] = F

        print(f"Trotter steps = {steps}, eps = {eps} → Fidelity = {100 * F:.6f}")

Trotter steps = 5, eps = 0.15 → Fidelity = 71.133564
Trotter steps = 5, eps = 0.15416666666666667 → Fidelity = 46.222766
Trotter steps = 5, eps = 0.15833333333333333 → Fidelity = 81.252635
Trotter steps = 5, eps = 0.1625 → Fidelity = 74.890755
Trotter steps = 5, eps = 0.16666666666666666 → Fidelity = 51.973997
Trotter steps = 5, eps = 0.17083333333333334 → Fidelity = 53.631650
Trotter steps = 5, eps = 0.175 → Fidelity = 74.406132
Trotter steps = 5, eps = 0.17916666666666667 → Fidelity = 47.778642
Trotter steps = 5, eps = 0.18333333333333332 → Fidelity = 76.088611
Trotter steps = 5, eps = 0.1875 → Fidelity = 78.330187
Trotter steps = 5, eps = 0.19166666666666665 → Fidelity = 70.177426
Trotter steps = 5, eps = 0.19583333333333333 → Fidelity = 71.570001
Trotter steps = 5, eps = 0.2 → Fidelity = 72.539854
Trotter steps = 5, eps = 0.20416666666666666 → Fidelity = 80.665634
Trotter steps = 5, eps = 0.20833333333333331 → Fidelity = 54.486987
Trotter steps = 5, eps = 0.2125 → Fidelity = 83.304

In [27]:
# -----------------------------------
# Filter eps values by fidelity
# -----------------------------------

FIDELITY_THRESHOLD = 0.80   # 80%

high_fidelity_eps = []

# Since you only have one Trotter step, access it directly
steps = troter_steps[0]  # 5 in your case

for eps, F in fidelity_results[steps].items():
    if F >= FIDELITY_THRESHOLD:
        high_fidelity_eps.append((eps, F))

# Sort by fidelity (highest first)
high_fidelity_eps.sort(key=lambda x: x[1], reverse=True)

# Print results
print(f"\nEpsilon values with fidelity ≥ {100*FIDELITY_THRESHOLD:.0f}% for {steps} Trotter steps:")
for eps, F in high_fidelity_eps:
    print(f"eps = {eps:.6f}, Fidelity = {100*F:.2f}%")


Epsilon values with fidelity ≥ 80% for 5 Trotter steps:
eps = 0.229167, Fidelity = 83.68%
eps = 0.212500, Fidelity = 83.30%
eps = 0.220833, Fidelity = 81.47%
eps = 0.158333, Fidelity = 81.25%
eps = 0.216667, Fidelity = 80.91%
eps = 0.233333, Fidelity = 80.79%
eps = 0.204167, Fidelity = 80.67%


In [28]:
gup_coeff = 0.001
filtered_eps = [eps for eps, F in high_fidelity_eps]
for eps in filtered_eps:

    theta = eps * 3

    A00 = (8 + np.cos(theta))/9 \
      + gup_coeff * ((14/27)*(np.cos(theta) - 1) + (10/27)*theta*np.sin(theta))

    A02 = (np.sqrt(2)/9)*(np.cos(theta) - 1) \
      + gup_coeff*np.sqrt(2)*((14/27)*(np.cos(theta) - 1) + (10/27)*theta*np.sin(theta))

    A11 = (1j/3)*np.sin(theta) \
      + (10j/9)*gup_coeff*(np.sin(theta) - theta*np.cos(theta))

    A20 = (np.sqrt(2)/9)*(np.cos(theta) - 1) \
      + gup_coeff*np.sqrt(2)*(1 + (8/27)*(np.cos(theta) - 1) + (10/27)*theta*np.sin(theta))

    A22 = (2/9)*(np.cos(theta) - 1) \
      + gup_coeff*((16/27)*(np.cos(theta) - 1) + (20/27)*theta*np.sin(theta))

    print(f"eps = {eps}")

    print("p00 =", abs(A00)**2)
    print("p02 =", abs(A02)**2)
    print("p11 =", abs(A11)**2)
    print("p20 =", abs(A20)**2)
    print("p22 =", abs(A22)**2)
    print()

eps = 0.22916666666666666
p00 = 0.9502413546478176
p02 = 0.0012697535358344887
p11 = 0.04479591395367556
p20 = 0.0011660856695892934
p22 = 0.0025293415786743663

eps = 0.2125
p00 = 0.9569044592849197
p02 = 0.0009491781039592192
p11 = 0.03939757588592698
p20 = 0.0008604127899414448
p22 = 0.0018907569189304787

eps = 0.22083333333333333
p00 = 0.9536234479449736
p02 = 0.0011010758878856938
p11 = 0.04207955294137116
p20 = 0.0010050021220647225
p22 = 0.0021933365276315267

eps = 0.15833333333333333
p00 = 0.9755952336772931
p02 = 0.00030148652550073815
p11 = 0.023251661182182234
p20 = 0.00025326691277404584
p22 = 0.0006005590509505833

eps = 0.21666666666666667
p00 = 0.9552767526082598
p02 = 0.0010230932959067428
p11 = 0.040733848042027836
p20 = 0.0009307094702419056
p22 = 0.002037995598951116

eps = 0.23333333333333334
p00 = 0.9485132318343462
p02 = 0.0013607042188985214
p11 = 0.04616488373690008
p20 = 0.0012531322443199995
p22 = 0.0027105149114066934

eps = 0.20416666666666666
p00 = 0.9600